#### Model with Gating at position G (at attention calculation step) :  As in Paper where stated best effective

In [ ]:
### Aplied gated atention at position G elementwise

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import random
import numpy as np
import torch

SEED = 12

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
torch.use_deterministic_algorithms(True)

In [ ]:
# Rope

def compute_rope_params(seq_len, head_dim, device=None):
    # x: (seq_len, dim)

    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    theta = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))

    pos = torch.arange(seq_len).float()

    angles = pos[:, None] * theta[None, :]


    angles = angles[None, None, :, :]

    return torch.cos(angles), torch.sin(angles)



# similar to sebastian
def apply_rope(x, cos, sin, offset=0):

    batch_size, num_heads, seq_len, head_dim = x.shape   # (batch_size, num_heads, seq_len, head_dim)

    assert head_dim % 2 == 0, "Head dimension must be even"

    cos_sel = cos[...,offset : offset + seq_len, :].to(x.device, x.dtype)  #(1,1,seq_len,head_dim//2)
    sin_sel = sin[..., offset : offset + seq_len, :].to(x.device, x.dtype)

    x_even = x[..., 0::2]  # (b,n_heads,seq_len,head_dim//2)
    x_odd  = x[..., 1::2]


    x_rot = torch.empty_like(x)

    x_rot[..., 0::2] = x_even * cos_sel - x_odd * sin_sel
    x_rot[..., 1::2] = x_even * sin_sel + x_odd * cos_sel

    return x_rot.to(dtype=x.dtype)

In [ ]:
import torch
from torch import nn


class causal_multi_head_transformer(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length



        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential(nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din),)
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.gate = nn.Linear(din, self.num_heads * self.head_dim)

        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, 2.0)  # sigmoid(2)  0.88
        # or for even closer to identity
        # nn.init.constant_(self.gate.bias, 4.0)  # sigmoid(4)  0.98

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
            self.q_norm = nn.RMSNorm(self.head_dim)
            self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)

    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN
        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)    # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)

        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)                             # shape: (b, num_heads, num_tokens, num_tokens)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        #gate applied elemnetwise

        gate_val = self.gate(x_norm)

        gate_val = gate_val.view(b, num_tokens, self.num_heads, self.head_dim)
        gate_val = gate_val.transpose(1,2)


        z = z * torch.sigmoid(gate_val)

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads * head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out

        elif self.pre_norm:                       #  Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    #  Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))

        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x, gate_val, sink_val

In [ ]:
class Gated_Transformer_LM(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()


    assert din == dout

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      gate_vals = []
      attn_sink_val = []

      x = self.embedding(inp)

      for layer in self.dstack:
        x, gate_val, sink_info = layer(x, self.cos, self.sin, start_pos)

        gate_vals.append(gate_val)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, gate_vals, attn_sink_val # Only return logits, as gate_val is not computed

### Model without Gating

In [ ]:
import torch
from torch import nn

class causal_multi_head_transformer_no_gate(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length

        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential( nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din), )
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
          self.q_norm = nn.RMSNorm(self.head_dim)
          self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)


    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN

        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        # Qk normalization L2 norm return unit vectors q/|q| = 1, k/|k| = 1 which scales the values in q.k = cos (theta) and range of values [-1, 1]
        # and no magnitude scale is taken in consideration in SDPA q.k/root(head_dim) which limits the range of values to [-1/root(head_dim), 1/root(head_dim)].
        # can add learnable paramereter Y which can change the scale [-1/root(head_dim) * y, 1/root(head_dim)] * y]
        # if self.qk_norm:
        #     q = q / (q.norm(dim=-1, keepdim=True) + 1e-6)
        #     k = k / (k.norm(dim=-1, keepdim=True) + 1e-6)

        # QK norm with RMSNorm (common with LLM models)

        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)



        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)
        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        # Gating mechanism removed

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads, head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out


        elif self.pre_norm:                       #  Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    #  Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))


        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x,  sink_val # Only return x, as gate_val is not computed

In [ ]:
class Transformer_No_Gate(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer_no_gate(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      attn_sink_val = []
      x = self.embedding(inp)

      for layer in self.dstack:
        x, sink_info = layer(x, self.cos, self.sin, start_pos)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, attn_sink_val # Only return logits, as gate_val is not computed


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import AutoTokenizer


# -----------------------------
# Load Dataset
# -----------------------------
# dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

texts = dataset["train"]["text"]
texts = [t for t in texts if len(t.strip()) > 0]


# -----------------------------
# Tokenizer (FAST)
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token


# -----------------------------
# FAST Tokenization (BATCHED)
# -----------------------------
encodings = tokenizer(
    texts,
    padding=False,
    truncation=False
)

# Flatten tokens + add EOS between docs
all_tokens = [
    token
    for ids in encodings["input_ids"]
    for token in (ids + [tokenizer.eos_token_id])
]

tokens = torch.tensor(all_tokens, dtype=torch.long)


# -----------------------------
# Lazy Dataset (NO stacking)
# -----------------------------
class WikiTextDataset(Dataset):

    def __init__(self, tokens, seq_len=64, stride=None):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = stride if stride is not None else seq_len

        self.num_samples = (len(tokens) - (seq_len + 1)) // self.stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        i = idx * self.stride
        chunk = self.tokens[i : i + self.seq_len + 1]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y


# -----------------------------
# Create Dataset
# -----------------------------
seq_len = 128

train_dataset = WikiTextDataset(
    tokens,
    seq_len=seq_len,
    stride=128
)

max_samples = 300000
train_dataset = Subset(train_dataset, range(min(max_samples, len(train_dataset))))


# # -----------------------------
# # DataLoader (OPTIMIZED)
# # -----------------------------
# dataloader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     num_workers=2,      #  parallel loading
#     pin_memory=True     #  faster GPU transfer
# )


# # -----------------------------
# # Example Batch
# # -----------------------------
# for x, y in dataloader:
#     print("Input shape:", x.shape)
#     print("Target shape:", y.shape)
#     break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1063 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,      #  parallel loading
    pin_memory=True     #  faster GPU transfer
)


len(dataloader)

18750

In [ ]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

50257


In [ ]:
outputs = tokenizer("a gaoal is good",  return_tensors="pt", add_special_tokens=True)['input_ids']
print(outputs)
print(tokenizer.decode(outputs, skip_special_tokens=False))

tensor([[  64,  308, 5488,  282,  318,  922]])
['a gaoal is good']


### Training without Gating

In [ ]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_no_gate(dataloader_dataset, qk_norm, pre_norm, post_norm, vocab_size, n_transformer, seed,
                          val_ratio=0.2):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))
    # ----------------------------------------------------------------------

    model_no_gate = Transformer_No_Gate(din=256, dout=256, context_length=128,  dropout=0.1, ff_dim=1024, num_heads=8,
                                        vocab_size = vocab_size, qk_norm = qk_norm, pre_norm = pre_norm, post_norm = post_norm, n_transformer = n_transformer).to(device)

    optimizer_no_gate = torch.optim.Adam(model_no_gate.parameters(), lr=1e-3)
    loss_fn_no_gate = nn.CrossEntropyLoss()

    loss_history_no_gate, max_act_history_no_gate, grad_norm_history_no_gate = [], [], []

    epoch_loss_no_gate = []
    val_epoch_loss_no_gate = []

    train_ppl_history_no_gate = []
    val_ppl_history_no_gate = []
    best_val_ppl_no_gate = float("inf")
    best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
    epochs_without_improvement_no_gate = 0
    patience_no_gate = 1

    print("\n--- Training without Gating ---")
    for epoch in range(10):
        model_no_gate.train()
        total_loss_no_gate=0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer_no_gate.zero_grad()
            out_no_gate, attn_sink_info = model_no_gate(xb)

            # Reshape for CrossEntropyLoss: (batch * seq, vocab)
            loss_no_gate = loss_fn_no_gate(out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
            loss_no_gate.backward()

            total_norm_no_gate = torch.nn.utils.clip_grad_norm_(model_no_gate.parameters(), max_norm=1.0)
            optimizer_no_gate.step()

            loss_history_no_gate.append(loss_no_gate.item())
            max_act_history_no_gate.append(out_no_gate.abs().max().item())
            grad_norm_history_no_gate.append(total_norm_no_gate.item())

            total_loss_no_gate += loss_no_gate.item()

            if step % 500 == 0:
              out_no_gate_mean = out_no_gate.mean().item()
              max_act = out_no_gate.abs().max().item()

              # Accessing sink info for the last layer: attn_sink_info[-1]
              # attn_sink_info is a list of tensors
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(f"Epoch {epoch} Step {step} | Loss={loss_no_gate.item():.4f} | "
                    f"MaxAct={max_act:.4f} | MeanAct={out_no_gate_mean:.4f} | "
                    f"GradNorm={total_norm_no_gate:.4f} | Sink Val={avg_sink_val:.4f}")



        avg_train_loss_no_gate = total_loss_no_gate / len(train_dataloader)
        epoch_loss_no_gate.append(avg_train_loss_no_gate)

        train_ppl_no_gate = math.exp(min(avg_train_loss_no_gate, 20))
        train_ppl_history_no_gate.append(train_ppl_no_gate)

        # ===================== Validation =====================
        model_no_gate.eval()
        total_val_loss_no_gate = 0.0

        with torch.no_grad():
            for step, (xb, yb) in enumerate(val_dataloader):
                xb, yb = xb.to(device), yb.to(device)

                val_out_no_gate, attn_sink_info = model_no_gate(xb)
                val_loss_no_gate = loss_fn_no_gate(val_out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
                total_val_loss_no_gate += val_loss_no_gate.item()

        avg_val_loss_no_gate = total_val_loss_no_gate / len(val_dataloader)
        val_epoch_loss_no_gate.append(avg_val_loss_no_gate)

        val_ppl_no_gate = math.exp(min(avg_val_loss_no_gate, 20))
        val_ppl_history_no_gate.append(val_ppl_no_gate)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss_no_gate:.4f} | Train PPL={train_ppl_no_gate:.2f} | "
            f"Val Loss={avg_val_loss_no_gate:.4f} | Val PPL={val_ppl_no_gate:.2f}\n"
        )

        if val_ppl_no_gate < best_val_ppl_no_gate:
            best_val_ppl_no_gate = val_ppl_no_gate
            best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
            epochs_without_improvement_no_gate = 0
        else:
            epochs_without_improvement_no_gate += 1
            if epochs_without_improvement_no_gate >= patience_no_gate:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    model_no_gate.load_state_dict(best_model_state_no_gate)

    return {
        "model_no_gate": model_no_gate,
        "train_loss_no_gate": epoch_loss_no_gate,
        "val_loss_no_gate": val_epoch_loss_no_gate,
        "train_ppl_no_gate": train_ppl_history_no_gate,
        "val_ppl_no_gate": val_ppl_history_no_gate,
    }

### config 1: pre norm =True, post_norm = False, qk norm = False

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0174 | MaxAct=3.2652 | MeanAct=0.0006 | GradNorm=1.7499 | Sink Val=0.0423
Epoch 0 Step 500 | Loss=5.9639 | MaxAct=15.5792 | MeanAct=-3.8247 | GradNorm=0.5696 | Sink Val=0.0326
Epoch 0 Step 1000 | Loss=5.6303 | MaxAct=16.9313 | MeanAct=-3.8176 | GradNorm=0.5666 | Sink Val=0.0309
Epoch 0 Step 1500 | Loss=5.3347 | MaxAct=18.1947 | MeanAct=-3.9398 | GradNorm=0.5899 | Sink Val=0.0308
Epoch 0 Step 2000 | Loss=5.5404 | MaxAct=18.7770 | MeanAct=-3.8532 | GradNorm=0.6247 | Sink Val=0.0277
Epoch 0 Step 2500 | Loss=5.0714 | MaxAct=19.5068 | MeanAct=-3.9133 | GradNorm=0.6444 | Sink Val=0.0319
Epoch 0 Step 3000 | Loss=4.9643 | MaxAct=20.5793 | MeanAct=-4.0544 | GradNorm=0.6073 | Sink Val=0.0291
Epoch 0 Step 3500 | Loss=4.7268 | MaxAct=20.9892 | MeanAct=-3.9564 | GradNorm=0.5978 | Sink Val=0.0291
Epoch 0 Step 4000 | Loss=4.8615 | MaxAct=19.9404 | MeanAct=-4.0368 | GradNorm=0.5842 |

In [ ]:
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])

print('\n')

print('Best Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

train_loss_no_gate:  [4.719488196150462, 4.0771412472724915, 3.8837324659824373, 3.771570744498571, 3.694350135056178, 3.636253528849284, 3.590219136444728, 3.5520580766677856, 3.5198732053438824, 3.4919002442995706]
val_loss_no_gate:  [4.2422461548487345, 4.017440444628398, 3.9163849721272785, 3.8563350107828778, 3.821792586135864, 3.795255835723877, 3.777996182378133, 3.7636579288482666, 3.7504323752085367, 3.7399949403127035]
train_ppl_no_gate:  [112.11085921264636, 58.97662902869534, 48.60529453306802, 43.448257376548646, 40.2194268977066, 37.949393753882525, 36.24201700311087, 34.885039743686335, 33.78014505042305, 32.84830824783357]
val_ppl_no_gate:  [69.56392784151853, 55.558718064588945, 50.2185746961658, 47.29170975957749, 45.68603110770787, 44.48961698305338, 43.728330287245534, 43.10581595030145, 42.53947103695511, 42.09777716286972]


Best Training PPL:  32.84830824783357
Best Validation PPL:  42.09777716286972


### config 2: pre norm = True, post_norm = False, qk norm = True

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0154 | MaxAct=3.2417 | MeanAct=0.0006 | GradNorm=1.7399 | Sink Val=0.0424
Epoch 0 Step 500 | Loss=5.9425 | MaxAct=15.8473 | MeanAct=-3.7970 | GradNorm=0.5702 | Sink Val=0.0307
Epoch 0 Step 1000 | Loss=5.5844 | MaxAct=17.2085 | MeanAct=-3.8148 | GradNorm=0.5575 | Sink Val=0.0336
Epoch 0 Step 1500 | Loss=5.3255 | MaxAct=18.1294 | MeanAct=-3.9266 | GradNorm=0.6195 | Sink Val=0.0321
Epoch 0 Step 2000 | Loss=5.5031 | MaxAct=18.9850 | MeanAct=-3.8773 | GradNorm=0.6118 | Sink Val=0.0333
Epoch 0 Step 2500 | Loss=5.0744 | MaxAct=19.3436 | MeanAct=-3.9459 | GradNorm=0.6240 | Sink Val=0.0328
Epoch 0 Step 3000 | Loss=4.9387 | MaxAct=19.8866 | MeanAct=-4.0712 | GradNorm=0.5835 | Sink Val=0.0313
Epoch 0 Step 3500 | Loss=4.7267 | MaxAct=20.6365 | MeanAct=-3.9599 | GradNorm=0.5826 | Sink Val=0.0331
Epoch 0 Step 4000 | Loss=4.8449 | MaxAct=20.0442 | MeanAct=-4.0510 | GradNorm=0.5788 |

In [ ]:
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])

print('\n')

print('Best Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

train_loss_no_gate:  [4.712412454271316, 4.066887979141871, 3.8727493121147156, 3.7605480034510292, 3.6832162594000497, 3.6249245700041453, 3.5788403319835664, 3.54066681543986, 3.5083493403116863, 3.480303757127126]
val_loss_no_gate:  [4.233359502919515, 4.0061597819646195, 3.9036323252360026, 3.8467472931543987, 3.8097984102884928, 3.7841302474975587, 3.7651725905100504, 3.7544829750696818, 3.740701077906291, 3.7284124894460042]
train_ppl_no_gate:  [111.32039158143758, 58.375015361547284, 48.07437602131129, 42.971968314467105, 39.77411243778626, 37.52189277663785, 35.831963561689754, 34.48990992917601, 33.39310161947779, 32.469583445443696]
val_ppl_no_gate:  [68.94847613083991, 54.93550066953137, 49.58222116577596, 46.840456899489865, 45.14133791973265, 43.997387075075984, 43.17115614877612, 42.712130864292654, 42.127514484015414, 41.61299463178775]


Best Training PPL:  32.469583445443696
Best Validation PPL:  41.61299463178775


### config 3: pre_norm = True, post norm = True, qk norm = True

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0188 | MaxAct=3.2044 | MeanAct=-0.0000 | GradNorm=3.1944 | Sink Val=0.0445
Epoch 0 Step 500 | Loss=6.0062 | MaxAct=15.4789 | MeanAct=-3.7519 | GradNorm=0.5892 | Sink Val=0.0367
Epoch 0 Step 1000 | Loss=5.6218 | MaxAct=17.2794 | MeanAct=-3.8463 | GradNorm=0.5826 | Sink Val=0.0332
Epoch 0 Step 1500 | Loss=5.3338 | MaxAct=18.2676 | MeanAct=-3.9184 | GradNorm=0.6271 | Sink Val=0.0336
Epoch 0 Step 2000 | Loss=5.5248 | MaxAct=18.9323 | MeanAct=-3.8643 | GradNorm=0.6264 | Sink Val=0.0319
Epoch 0 Step 2500 | Loss=5.0269 | MaxAct=19.2536 | MeanAct=-3.8953 | GradNorm=0.5981 | Sink Val=0.0332
Epoch 0 Step 3000 | Loss=4.9775 | MaxAct=20.5794 | MeanAct=-3.9992 | GradNorm=0.6094 | Sink Val=0.0313
Epoch 0 Step 3500 | Loss=4.7461 | MaxAct=20.1889 | MeanAct=-4.0405 | GradNorm=0.5937 | Sink Val=0.0324
Epoch 0 Step 4000 | Loss=4.8475 | MaxAct=20.6911 | MeanAct=-4.0336 | GradNorm=0.5790 

In [ ]:
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])

print('\n')

print('Best Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

train_loss_no_gate:  [4.72731697529157, 4.071902536694209, 3.8705114835898082, 3.752848777929942, 3.6707820029735565, 3.6080569364706676, 3.5579122698783876, 3.5156624771118166, 3.4795779829661053, 3.4476841219266254]
val_loss_no_gate:  [4.246080546442668, 4.013217852783203, 3.907682739384969, 3.8446110368092854, 3.808924256960551, 3.7790907812754315, 3.75855570081075, 3.7451468229929605, 3.7347536520004274, 3.720465465418498]
train_ppl_no_gate:  [112.9919949770079, 58.66847540689966, 47.966914096859945, 42.64238782466129, 39.282612972366636, 36.89429516318929, 35.089862457409296, 33.63820508500476, 32.44602641033763, 31.427525648614736]
val_ppl_no_gate:  [69.8311752197905, 55.32461089212825, 49.7834569649668, 46.74050048051082, 45.101894711180066, 43.77622147539781, 42.886440375595704, 42.315219609956635, 41.8777078027312, 41.28360572790896]


Best Training PPL:  31.427525648614736
Best Validation PPL:  41.28360572790896


### Training with Gating

In [ ]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_with_gate(dataloader_dataset, qk_norm, pre_norm, post_norm,
                             vocab_size, n_transformer, seed,
                             val_ratio=0.2, patience=1):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))

    # ----------------------------------------------------------------------

    model = Gated_Transformer_LM(
      din=256, dout=256, context_length=128, dropout=0.1,
      ff_dim=1024, num_heads=8,
      vocab_size=vocab_size, qk_norm=qk_norm,
      pre_norm=pre_norm, post_norm=post_norm,
      n_transformer=n_transformer
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    loss_history, gate_mean_history, max_act_history, grad_norm_history = [], [], [], []

    print("\n--- Training with Gating ---")
    gated_epoch_loss = []

    train_ppl_history = []
    val_ppl_history = []
    val_epoch_loss = []

    best_val_ppl = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    # -------------------------------------------------------------------

    for epoch in range(10):
        model.train()
        total_loss = 0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, gate_vals, attn_sink_info = model(xb)
            loss = loss_fn(out.view(-1, vocab_size), yb.view(-1))
            loss.backward()

            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Existing statistics (UNCHANGED)
            gates = [torch.sigmoid(g) for g in gate_vals]
            layer_means = [g.mean().item() for g in gates]
            layer_std = [g.std().item() for g in gates]
            layer_sparsity = [(g < 0.1).float().mean().item() for g in gates]

            all_gates_combined = torch.stack(gate_vals)
            g = torch.sigmoid(all_gates_combined)

            gate_mean = g.mean().item()
            sparsity = (g < 0.1).float().mean()
            head_mean = g.mean(dim=(0, 1, 3, 4))

            dead = torch.mean(torch.stack([
              (g < 0.01).float().mean() for g in gates
            ])).item()

            open_ = torch.mean(torch.stack([
              (g > 0.99).float().mean() for g in gates
            ])).item()

            loss_history.append(loss.item())
            gate_mean_history.append(gate_mean)
            max_act_history.append(out.abs().max().item())
            grad_norm_history.append(total_norm.item())
            total_loss += loss.item()

            if step % 500 == 0:
              out_mean = out.mean().item()
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(
                  f"Epoch {epoch} Step {step} | Loss={loss.item():.4f} | "
                  f"gate_mean={gate_mean:.4f} | "
                  f"LayerMeans={[round(m, 3) for m in layer_means]} | "
                  f"LayerStd={[round(s, 3) for s in layer_std]} | "
                  f"LayerSparsity={[round(s, 3) for s in layer_sparsity]} | "
                  f"MaxAct={out.abs().max():.4f} | MeanAct={out_mean:.4f} | GradNorm={total_norm:.4f} | "
                  f"Sparsity={sparsity.item():.4f} | HeadMeanAvg={head_mean.mean().item():.4f} | "
                  f"Sink Val={avg_sink_val:.4f} | Dead={dead:.3f} | Open={open_:.3f}"
              )

        avg_train_loss = total_loss / len(train_dataloader)
        train_ppl = math.exp(min(avg_train_loss, 20))
        train_ppl_history.append(train_ppl)
        # -------------------------------------------------------------

        gated_epoch_loss.append(avg_train_loss)

        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_dataloader:
                xb, yb = xb.to(device), yb.to(device)
                val_out, _, _ = model(xb)
                val_loss = loss_fn(val_out.view(-1, vocab_size), yb.view(-1))
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_ppl = math.exp(min(avg_val_loss, 20))

        val_epoch_loss.append(avg_val_loss)
        val_ppl_history.append(val_ppl)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss:.4f} | Train PPL={train_ppl:.2f} | "
            f"Val Loss={avg_val_loss:.4f} | Val PPL={val_ppl:.2f}\n"
        )
        # -------------------------------------------------------------

        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break
        # -------------------------------------------------------------

    model.load_state_dict(best_model_state)
    # --------------------------------------------------------------

    return {
    "model": model,
    "train_loss": gated_epoch_loss,
    "val_loss": val_epoch_loss,
    "train_ppl": train_ppl_history,
    "val_ppl": val_ppl_history,
    "step_loss": loss_history,
    "gate_mean_history": gate_mean_history,
    "max_activation": max_act_history,
    "grad_norm": grad_norm_history,
    }

### config 1: pre norm =True, post_norm = False, qk norm = False

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9586 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3624 | MeanAct=-0.0002 | GradNorm=1.7579 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0424 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9388 | gate_mean=0.5522 | LayerMeans=[0.841, 0.442, 0.468, 0.501, 0.496, 0.561, 0.544, 0.565] | LayerStd=[0.113, 0.244, 0.265, 0.281, 0.289, 0.296, 0.297, 0.289] | LayerSparsity=[0.001, 0.064, 0.085, 0.082, 0.104, 0.074, 0.088, 0.068] | MaxAct=15.7813 | MeanAct=-3.8000 | GradNorm=0.5575 | Sparsity=0.0707 | HeadMeanAvg=0.5522 | Sink Val=0.0365 | Dead=0.001 | Open=0.002
Epoch 0 Step 1000 | Loss=5.5522 | gate_mean=0.3978 | LayerMeans=[0.817, 0.266, 0.283, 0.317, 0.304, 0.382, 0.379, 0.434] | LayerStd=[0.144, 0.242, 0.262,

In [ ]:
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

train_loss:  [4.695832725556691, 4.0634013029416405, 3.8667303202788035, 3.7520635724862417, 3.6729804850419363, 3.61296180173556, 3.5650904848416647, 3.525353162463506, 3.491419407304128, 3.4621813180446623]
val_loss_no_gate:  [4.227023003514608, 3.9931513382593793, 3.8884021489461262, 3.828101762072245, 3.79062712504069, 3.760871481513977, 3.7387651096343992, 3.7242687721888226, 3.715236792119344, 3.7007392265955605]
train_ppl_no_gate:  [109.4899457664804, 58.17183500269595, 47.78588582554666, 42.608917931699715, 39.36907009497943, 37.07570123248768, 35.34265123380086, 33.96576693225305, 32.83251736271542, 31.88645520968215]
val_ppl_no_gate:  [68.51296541674768, 54.22550329419145, 48.83279661678855, 45.97518351334228, 44.28416327733402, 42.9858710520326, 42.04603584807603, 41.44091891059397, 41.06831058542449, 40.477215137227994]


Best Training PPL:  31.88645520968215
Best Validation PPL:  40.477215137227994


### config 2: pre norm =True, post_norm = False, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9605 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3866 | MeanAct=-0.0002 | GradNorm=1.7546 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0422 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.8988 | gate_mean=0.5842 | LayerMeans=[0.84, 0.497, 0.521, 0.527, 0.536, 0.576, 0.584, 0.591] | LayerStd=[0.115, 0.247, 0.27, 0.285, 0.288, 0.295, 0.303, 0.292] | LayerSparsity=[0.001, 0.041, 0.059, 0.078, 0.081, 0.074, 0.081, 0.065] | MaxAct=15.7915 | MeanAct=-3.8140 | GradNorm=0.5501 | Sparsity=0.0599 | HeadMeanAvg=0.5842 | Sink Val=0.0319 | Dead=0.001 | Open=0.003
Epoch 0 Step 1000 | Loss=5.5493 | gate_mean=0.4285 | LayerMeans=[0.812, 0.313, 0.334, 0.353, 0.346, 0.403, 0.419, 0.447] | LayerStd=[0.151, 0.267, 0.283, 0

In [ ]:
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

train_loss:  [4.677497733640671, 4.054813833030065, 3.8610034872214, 3.7481135477224985, 3.669686558834712, 3.610477603260676, 3.5632950085163118, 3.52421307776769, 3.490927352841695, 3.462142832740148]
val_loss_no_gate:  [4.213144381904602, 3.988619053586324, 3.884658376757304, 3.822900394821167, 3.783943663978577, 3.758944452857971, 3.740096597099304, 3.725758367093404, 3.7143018840789797, 3.702404843711853]
train_ppl_no_gate:  [107.5007402553912, 57.67442492372607, 47.51300614889385, 42.44094362046201, 39.23960462514932, 36.98371213892089, 35.279251273831, 33.927065147040466, 32.81636595004742, 31.885228073357034]
val_ppl_no_gate:  [67.56866782602124, 53.980293975689705, 48.650319540344746, 45.73667053398524, 43.989178653571415, 42.903115808184005, 42.10205690514786, 41.502695091563666, 41.02993343400303, 40.544690858338704]


Best Training PPL:  31.885228073357034
Best Validation PPL:  40.544690858338704


### config 3: post_norm =True, post_norm = True, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9528 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.2551 | MeanAct=0.0005 | GradNorm=3.6987 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0443 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0016 | gate_mean=0.7299 | LayerMeans=[0.839, 0.652, 0.696, 0.717, 0.748, 0.69, 0.75, 0.747] | LayerStd=[0.128, 0.267, 0.275, 0.266, 0.242, 0.286, 0.241, 0.253] | LayerSparsity=[0.001, 0.029, 0.037, 0.032, 0.016, 0.045, 0.015, 0.021] | MaxAct=15.8380 | MeanAct=-3.7533 | GradNorm=0.6440 | Sparsity=0.0244 | HeadMeanAvg=0.7299 | Sink Val=0.0343 | Dead=0.000 | Open=0.014
Epoch 0 Step 1000 | Loss=5.6107 | gate_mean=0.6136 | LayerMeans=[0.82, 0.504, 0.563, 0.572, 0.621, 0.559, 0.617, 0.651] | LayerStd=[0.16, 0.316, 0.33, 0.336

In [ ]:
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

train_loss:  [4.693947413667043, 4.039166353750229, 3.839146086661021, 3.7214957962989805, 3.639353790807724, 3.576455140765508, 3.5259627953211465, 3.483648306798935, 3.447443167289098, 3.4158053368409473]
val_loss_no_gate:  [4.205860771814982, 3.97514150651296, 3.8727877668380737, 3.80864072303772, 3.769423676363627, 3.7443810899098713, 3.7228513161977133, 3.7109914608637493, 3.7012292214075724, 3.689115228907267]
train_ppl_no_gate:  [109.28371753328477, 56.778989478605126, 46.48576267417898, 41.32616342223656, 38.06722938195846, 35.74659932236434, 33.98647989281907, 32.5783613853122, 31.41995395281793, 30.441455183898167]
val_ppl_no_gate:  [67.07831194364724, 53.257652677261454, 48.076224743687625, 45.089108606778325, 43.35507108207666, 42.28282984892153, 41.38221984330746, 40.89433156347114, 40.49705362263046, 40.00943210361568]


Best Training PPL:  30.441455183898167
Best Validation PPL:  40.00943210361568


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
